In [ ]:
%pip install pandas requests tqdm


   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ---------------------------------------- 0/6 [urllib3]
   ------ --------------------------------- 1/6 [tqdm]
   ------ --------------------------------- 1/6 [tqdm]
   ------ --------------------------------- 1/6 [tqdm]
   ------ --------------------------------- 1/6 [tqdm]
   ------------- -------------------------- 2/6 [idna]
   -------------------- ------------------- 3/6 [charset_normalizer]
   -------------------- ------------------- 3/6 [charset_normalizer]
   -------------------------- ------------- 4/6 [certifi]
   --------------------------------- ------ 5/6 [requests]
   --------------------------------- ------ 5/6 [requests]
   ---------------------------------------- 6/6 [requests]

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import pandas as pd
import requests
from tqdm import tqdm

INPUT_CSV = "plants.csv"
OUTPUT_CSV = "plants_with_taxonomy.csv"

df = pd.read_csv(INPUT_CSV)

def get_taxonomy(name):
    try:
        r = requests.get(
            "https://api.gbif.org/v1/species/match",
            params={"name": name},
            timeout=30
        )

        if r.status_code != 200:
            return {}

        data = r.json()

        return {
            "kingdom": data.get("kingdom"),
            "phylum": data.get("phylum"),
            "class": data.get("class"),
            "order": data.get("order"),
            "family": data.get("family"),
        }

    except Exception:
        return {}

unique_names = df["taxon_name"].dropna().unique()

taxonomy_rows = []

for name in tqdm(unique_names):
    row = {"taxon_name": name}
    row.update(get_taxonomy(name))
    taxonomy_rows.append(row)

taxonomy_df = pd.DataFrame(taxonomy_rows)

result = df.merge(
    taxonomy_df,
    on="taxon_name",
    how="left"
)

result.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {OUTPUT_CSV}")

100%|██████████| 11790/11790 [32:51<00:00,  5.98it/s]

Saved plants_with_taxonomy.csv
